In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
import os
import json
import logging
from rdkit import Chem

logger = logging.getLogger(__name__)
def _score_benchmark(args):
    """Worker function — scores one benchmark against the training SMILES."""
    benchmark_name, wrapped_objective, smiles_list, n, cache_dir = args
    cache_path = os.path.join(cache_dir, f"{benchmark_name}.json")

    if os.path.exists(cache_path):
        return benchmark_name, None  # already cached

    scores = wrapped_objective.score_list(smiles_list)
    scored = sorted(zip(smiles_list, scores), key=lambda x: x[1], reverse=True)
    top = [{"smiles": smi, "score": score} for smi, score in scored[:n]]

    with open(cache_path, "w") as f:
        json.dump({"benchmark": benchmark_name, "n": n, "top": top}, f)

    return benchmark_name, scored[0][1]  # return name + best score for logging


def precompute_top_n_from_training_set(
        training_set_path,
        cache_dir="guacamol_cache",
        n=1000,
        benchmark_version="v2",
        sample_size=None,
        seed=42,
        n_jobs=-1):

    from guacamol.benchmark_suites import goal_directed_benchmark_suite
    import multiprocessing

    os.makedirs(cache_dir, exist_ok=True)

    with open(training_set_path) as f:
        smiles_list = [line.strip() for line in f if line.strip()]
    logger.info(f"Loaded {len(smiles_list)} training molecules")

    if sample_size is not None and sample_size < len(smiles_list):
        import random
        random.seed(seed)
        smiles_list = random.sample(smiles_list, sample_size)
        logger.info(f"Subsampled to {len(smiles_list)} molecules")

    benchmarks = goal_directed_benchmark_suite(version_name=benchmark_version)

    args = [
        (b.name, b.wrapped_objective, smiles_list, n, cache_dir)
        for b in benchmarks
    ]

    if n_jobs == 1:
        results = [_score_benchmark(a) for a in args]
    else:
        n_jobs = multiprocessing.cpu_count() if n_jobs == -1 else n_jobs
        with multiprocessing.Pool(processes=n_jobs) as pool:
            results = pool.map(_score_benchmark, args)

    for name, best in results:
        if best is None:
            logger.info(f"'{name}': already cached, skipped")
        else:
            logger.info(f"'{name}': done, best score={best:.4f}")

    logger.info("Done precomputing all benchmarks.")



    
precompute_top_n_from_training_set(
    training_set_path="data/guacamol_v1_train.smiles",
    cache_dir="guacamol_cache",
    n=1000,
)